# Matched Training-Size Control Experiment

This notebook examines whether the observed cross-dataset performance gap can be explained by differences in training-set size. The BDNeuro training set is reduced from 4,160 to 987 images to match the PMRAM training-set size while preserving BDNeuro class proportions.

In [2]:
# ============================================================
# CONTROL EXPERIMENT
# Matched Training Size
# BDNeuro (987 train images) -> PMRAM
# ============================================================

import torch
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split


# ------------------------------------------------------------
# 1. Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

csv_path = "final_four_class_clean_dataset.csv"

df = pd.read_csv(csv_path)

print("\nTotal dataset size:", len(df))

print("\nDataset counts:")
print(df["dataset"].value_counts())


# ------------------------------------------------------------
# 3. Separate datasets
# ------------------------------------------------------------

bdneuro_df = df[
    df["dataset"] == "BDNeuro"
].copy()

pmram_df = df[
    df["dataset"] == "PMRAM"
].copy()


# ------------------------------------------------------------
# 4. Original BDNeuro splits
# ------------------------------------------------------------

bd_train_full_df = bdneuro_df[
    bdneuro_df["original_split"] == "train"
].copy()

bd_val_df = bdneuro_df[
    bdneuro_df["original_split"] == "val"
].copy()

bd_test_df = bdneuro_df[
    bdneuro_df["original_split"] == "test"
].copy()


print("\n" + "=" * 60)
print("ORIGINAL BDNEURO")
print("=" * 60)

print("Full Train:", len(bd_train_full_df))
print("Validation:", len(bd_val_df))
print("Test:", len(bd_test_df))

print("\nOriginal BDNeuro train class distribution:")
print(
    bd_train_full_df["class_label"].value_counts()
)


# ------------------------------------------------------------
# 5. Create matched-size BDNeuro training set
# ------------------------------------------------------------
# PMRAM train had 987 images.
#
# We randomly select exactly 987 images from the ORIGINAL
# BDNeuro training set while preserving class proportions.
# ------------------------------------------------------------

matched_train_size = 987

bd_train_matched_df, _ = train_test_split(
    bd_train_full_df,
    train_size=matched_train_size,
    stratify=bd_train_full_df["class_label"],
    random_state=42
)


# ------------------------------------------------------------
# 6. Check matched training set
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MATCHED-SIZE BDNEURO TRAINING SET")
print("=" * 60)

print(
    "Matched BDNeuro Train:",
    len(bd_train_matched_df)
)

print("\nClass counts:")

print(
    bd_train_matched_df[
        "class_label"
    ].value_counts()
)


print("\nClass proportions:")

print(
    bd_train_matched_df[
        "class_label"
    ].value_counts(
        normalize=True
    ).round(4)
)


# ------------------------------------------------------------
# 7. PMRAM reference
# ------------------------------------------------------------

# Recreate the exact PMRAM 70/15/15 split used previously

pm_train_df, pm_temp_df = train_test_split(
    pmram_df,
    test_size=0.30,
    stratify=pmram_df["class_label"],
    random_state=42
)

pm_val_df, pm_test_df = train_test_split(
    pm_temp_df,
    test_size=0.50,
    stratify=pm_temp_df["class_label"],
    random_state=42
)


print("\n" + "=" * 60)
print("PMRAM REFERENCE")
print("=" * 60)

print(
    "PMRAM Train:",
    len(pm_train_df)
)

print("\nPMRAM train class counts:")

print(
    pm_train_df[
        "class_label"
    ].value_counts()
)


# ------------------------------------------------------------
# 8. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL CHECK")
print("=" * 60)

print(
    "Original BDNeuro Train :",
    len(bd_train_full_df)
)

print(
    "Matched BDNeuro Train :",
    len(bd_train_matched_df)
)

print(
    "PMRAM Train :",
    len(pm_train_df)
)

print()

if len(bd_train_matched_df) == len(pm_train_df):
    print("SUCCESS: Training-set sizes are exactly matched.")
else:
    print("WARNING: Training-set sizes do not match.")


Device: cpu

Total dataset size: 7351

Dataset counts:
dataset
BDNeuro    5941
PMRAM      1410
Name: count, dtype: int64

ORIGINAL BDNEURO
Full Train: 4160
Validation: 892
Test: 889

Original BDNeuro train class distribution:
class_label
glioma        1534
pituitary     1040
meningioma    1035
no_tumor       551
Name: count, dtype: int64

MATCHED-SIZE BDNEURO TRAINING SET
Matched BDNeuro Train: 987

Class counts:
class_label
glioma        364
pituitary     247
meningioma    245
no_tumor      131
Name: count, dtype: int64

Class proportions:
class_label
glioma        0.3688
pituitary     0.2503
meningioma    0.2482
no_tumor      0.1327
Name: proportion, dtype: float64

PMRAM REFERENCE
PMRAM Train: 987

PMRAM train class counts:
class_label
pituitary     260
glioma        259
no_tumor      239
meningioma    229
Name: count, dtype: int64

FINAL CHECK
Original BDNeuro Train : 4160
Matched BDNeuro Train : 987
PMRAM Train : 987

SUCCESS: Training-set sizes are exactly matched.


## 1. Matched-Size Cross-Dataset Evaluation

A pretrained ResNet18 is fine-tuned using the 987-image BDNeuro training subset. The model is evaluated on the original BDNeuro internal test set and the complete PMRAM dataset as an external test set.

In [3]:
# ============================================================
# MATCHED-SIZE CONTROL EXPERIMENT
# ResNet18
# BDNeuro (987 training images) -> PMRAM
# ============================================================

import os
import copy
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# 1. SETTINGS
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

BATCH_SIZE = 32
NUM_EPOCHS = 5
LEARNING_RATE = 1e-4
NUM_CLASSES = 4

# Keep CPU usage conservative because the laptop kernel
# has crashed before.
NUM_WORKERS = 0

torch.manual_seed(42)
np.random.seed(42)


# ============================================================
# 2. CLASS MAPPING
# ============================================================

class_to_idx = {
    "glioma": 0,
    "meningioma": 1,
    "no_tumor": 2,
    "pituitary": 3
}

idx_to_class = {
    0: "glioma",
    1: "meningioma",
    2: "no_tumor",
    3: "pituitary"
}


# ============================================================
# 3. TRANSFORMS
# ============================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 4. DATASET CLASS
# ============================================================

class BrainMRIDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = row["filepath"]
        label_name = row["class_label"]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[label_name]

        return image, label


# ============================================================
# 5. CREATE DATASETS
# ============================================================

train_dataset = BrainMRIDataset(
    bd_train_matched_df,
    transform=train_transform
)

val_dataset = BrainMRIDataset(
    bd_val_df,
    transform=eval_transform
)

bd_test_dataset = BrainMRIDataset(
    bd_test_df,
    transform=eval_transform
)

# IMPORTANT:
# Use ALL PMRAM images as the independent external dataset,
# matching the original cross-dataset evaluation.
pm_external_dataset = BrainMRIDataset(
    pmram_df,
    transform=eval_transform
)


# ============================================================
# 6. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

bd_test_loader = DataLoader(
    bd_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

pm_external_loader = DataLoader(
    pm_external_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


print("\nDataset sizes:")
print("BDNeuro Matched Train :", len(train_dataset))
print("BDNeuro Validation :", len(val_dataset))
print("BDNeuro Internal Test :", len(bd_test_dataset))
print("PMRAM External :", len(pm_external_dataset))


# ============================================================
# 7. BUILD RESNET18
# ============================================================

print("\nLoading ResNet18...")

model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 8. TRAINING
# ============================================================

best_val_accuracy = 0.0
best_model_state = copy.deepcopy(model.state_dict())

history = []

print("\n" + "=" * 65)
print("TRAINING STARTED")
print("=" * 65)

training_start = time.time()


for epoch in range(NUM_EPOCHS):

    epoch_start = time.time()

    # ---------------- TRAIN ----------------

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        train_loss += (
            loss.item() * images.size(0)
        )

        predictions = outputs.argmax(dim=1)

        train_correct += (
            predictions == labels
        ).sum().item()

        train_total += labels.size(0)


    train_loss = train_loss / train_total

    train_accuracy = (
        train_correct / train_total
    )


    # ---------------- VALIDATION ----------------

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            val_loss += (
                loss.item() * images.size(0)
            )

            predictions = outputs.argmax(dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)


    val_loss = val_loss / val_total

    val_accuracy = (
        val_correct / val_total
    )


    # ---------------- SAVE BEST MODEL ----------------

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        best_model_state = copy.deepcopy(
            model.state_dict()
        )


    epoch_minutes = (
        time.time() - epoch_start
    ) / 60


    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy
    })


    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {epoch_minutes:.1f} min"
    )


total_minutes = (
    time.time() - training_start
) / 60


print("\nTraining completed.")
print(
    f"Total training time: "
    f"{total_minutes:.1f} minutes"
)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f}"
)


# ============================================================
# 9. LOAD BEST MODEL
# ============================================================

model.load_state_dict(
    best_model_state
)

model.eval()


# ============================================================
# 10. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    model,
    loader,
    dataset_name
):

    all_labels = []
    all_predictions = []

    model.eval()

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            predictions = outputs.argmax(dim=1)

            all_labels.extend(
                labels.numpy()
            )

            all_predictions.extend(
                predictions.cpu().numpy()
            )


    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )


    print("\n" + "=" * 65)
    print(dataset_name)
    print("=" * 65)

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"Macro-F1 : {macro_f1:.4f}"
    )


    print("\nClassification Report:")

    print(
        classification_report(
            all_labels,
            all_predictions,
            target_names=[
                "Glioma",
                "Meningioma",
                "No Tumor",
                "Pituitary"
            ],
            digits=4,
            zero_division=0
        )
    )


    print("Confusion Matrix:")

    print(
        confusion_matrix(
            all_labels,
            all_predictions
        )
    )


    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "labels": all_labels,
        "predictions": all_predictions
    }


# ============================================================
# 11. INTERNAL BDNEURO TEST
# ============================================================

bd_internal_results = evaluate_model(
    model,
    bd_test_loader,
    "BDNeuro INTERNAL TEST"
)


# ============================================================
# 12. EXTERNAL PMRAM TEST
# ============================================================

pm_external_results = evaluate_model(
    model,
    pm_external_loader,
    "PMRAM EXTERNAL TEST"
)


# ============================================================
# 13. SUMMARY
# ============================================================

summary = pd.DataFrame({

    "Experiment": [
        "BDNeuro Matched Internal",
        "BDNeuro Matched -> PMRAM"
    ],

    "Training_Size": [
        len(bd_train_matched_df),
        len(bd_train_matched_df)
    ],

    "Evaluation_Size": [
        len(bd_test_df),
        len(pmram_df)
    ],

    "Accuracy": [
        bd_internal_results["accuracy"],
        pm_external_results["accuracy"]
    ],

    "Macro_F1": [
        bd_internal_results["macro_f1"],
        pm_external_results["macro_f1"]
    ]
})


print("\n" + "=" * 65)
print("FINAL MATCHED-SIZE RESULTS")
print("=" * 65)

print(
    summary.to_string(
        index=False
    )
)


# ============================================================
# 14. SAVE RESULTS
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(
    "resnet18_bdneuro_matched_training_history.csv",
    index=False
)

summary.to_csv(
    "resnet18_bdneuro_matched_results.csv",
    index=False
)

torch.save(
    best_model_state,
    "resnet18_bdneuro_matched_987.pth"
)


print("\nSaved:")
print(
    "resnet18_bdneuro_matched_training_history.csv"
)

print(
    "resnet18_bdneuro_matched_results.csv"
)

print(
    "resnet18_bdneuro_matched_987.pth"
)

print("\nEXPERIMENT FINISHED SUCCESSFULLY.")


Device: cpu

Dataset sizes:
BDNeuro Matched Train : 987
BDNeuro Validation : 892
BDNeuro Internal Test : 889
PMRAM External : 1410

Loading ResNet18...

TRAINING STARTED
Epoch 1/5 | Train Acc: 0.7315 | Val Acc: 0.8240 | Time: 3.8 min
Epoch 2/5 | Train Acc: 0.9088 | Val Acc: 0.8845 | Time: 3.3 min
Epoch 3/5 | Train Acc: 0.9676 | Val Acc: 0.9025 | Time: 3.3 min
Epoch 4/5 | Train Acc: 0.9777 | Val Acc: 0.9058 | Time: 3.3 min
Epoch 5/5 | Train Acc: 0.9838 | Val Acc: 0.9137 | Time: 3.3 min

Training completed.
Total training time: 16.9 minutes
Best validation accuracy: 0.9137

BDNeuro INTERNAL TEST
Accuracy : 0.9066
Macro-F1 : 0.9047

Classification Report:
              precision    recall  f1-score   support

      Glioma     0.9252    0.9055    0.9153       328
  Meningioma     0.8438    0.8552    0.8494       221
    No Tumor     0.8720    0.9237    0.8971       118
   Pituitary     0.9635    0.9505    0.9569       222

    accuracy                         0.9066       889
   macro avg 

## 2. Per-Class External Performance

Per-class performance is examined on PMRAM to identify which tumor categories remain most affected under cross-dataset evaluation after controlling for training-set size.

In [4]:
# ============================================================
# PER-CLASS ANALYSIS
# ResNet18 Matched BDNeuro (987) -> PMRAM
# No retraining
# ============================================================

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Get existing external predictions
# ------------------------------------------------------------

y_true = np.array(
    pm_external_results["labels"]
)

y_pred = np.array(
    pm_external_results["predictions"]
)


# ------------------------------------------------------------
# Calculate per-class metrics
# ------------------------------------------------------------

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        zero_division=0
    )
)


class_names = [
    "Glioma",
    "Meningioma",
    "No Tumor",
    "Pituitary"
]


per_class_matched = pd.DataFrame({

    "Class": class_names,

    "Precision": precision,

    "Recall": recall,

    "F1": f1,

    "Support": support
})


# ------------------------------------------------------------
# Number of correct / incorrect predictions
# ------------------------------------------------------------

per_class_matched["Correct"] = [
    ((y_true == i) & (y_pred == i)).sum()
    for i in range(4)
]

per_class_matched["Errors"] = (
    per_class_matched["Support"]
    - per_class_matched["Correct"]
)


# ------------------------------------------------------------
# Round metrics
# ------------------------------------------------------------

for col in [
    "Precision",
    "Recall",
    "F1"
]:

    per_class_matched[col] = (
        per_class_matched[col].round(4)
    )


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("=" * 75)

print(
    "RESNET18: BDNEURO-987 -> PMRAM"
)

print(
    "PER-CLASS EXTERNAL PERFORMANCE"
)

print("=" * 75)

print(
    per_class_matched.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# Identify weakest class
# ------------------------------------------------------------

weakest = (
    per_class_matched
    .sort_values("F1")
    .iloc[0]
)


print("\n" + "=" * 75)

print("WEAKEST EXTERNAL CLASS")

print("=" * 75)

print(
    "Class:",
    weakest["Class"]
)

print(
    "F1:",
    weakest["F1"]
)

print(
    "Recall:",
    weakest["Recall"]
)

print(
    "Errors:",
    int(weakest["Errors"])
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

per_class_matched.to_csv(
    "resnet18_bd987_to_pmram_per_class.csv",
    index=False
)


print(
    "\nSaved: "
    "resnet18_bd987_to_pmram_per_class.csv"
)


RESNET18: BDNEURO-987 -> PMRAM
PER-CLASS EXTERNAL PERFORMANCE
     Class  Precision  Recall     F1  Support  Correct  Errors
    Glioma     0.9517  0.8514 0.8987      370      315      55
Meningioma     0.7975  0.5780 0.6702      327      189     138
  No Tumor     0.8568  0.9824 0.9153      341      335       6
 Pituitary     0.7805  0.9462 0.8554      372      352      20

WEAKEST EXTERNAL CLASS
Class: Meningioma
F1: 0.6702
Recall: 0.578
Errors: 138

Saved: resnet18_bd987_to_pmram_per_class.csv


## 3. Key Findings

- Reducing the BDNeuro training set to 987 images resulted in 90.66% internal accuracy and 84.47% external accuracy on PMRAM.
- Cross-dataset performance remained lower than internal performance despite controlling the training-set size.
- Meningioma showed the weakest external performance, with a recall of 57.80% and F1-score of 67.02%.
- These results indicate that training-set size alone does not explain the observed cross-dataset performance degradation.



